In [1]:
import json
import re
from collections import defaultdict

In [2]:
def print_dataset_stats(dataset, label="dataset", mode="total"):
    # Prints #users, #conversations, #messages, #thoughts, #llms — globally, or per-model when mode="modelwise"
    
    users_by_model = defaultdict(set)
    convs_by_model = defaultdict(int)
    msgs_by_model = defaultdict(int)
    thoughts_by_model = defaultdict(int)

    for key, conv in dataset.items():
        model = conv.get("model_name") or "(unknown)"
        m = re.match(r"(user\d+)_task\d+_conversation\d+", key)
        if m:
            users_by_model[model].add(m.group(1))
        convs_by_model[model] += 1
        for msg in conv.get("messages", []):
            msgs_by_model[model] += 1
            thoughts_by_model[model] += len(msg.get("reasons") or [])
            thoughts_by_model[model] += len(msg.get("reactions") or [])

    total_users = set().union(*users_by_model.values()) if users_by_model else set()
    total_convs = sum(convs_by_model.values())
    total_msgs = sum(msgs_by_model.values())
    total_thoughts = sum(thoughts_by_model.values())

    if mode == "modelwise":
        models = sorted(convs_by_model, key=lambda m: (-len(users_by_model[m]), -convs_by_model[m], m))
        print(f"=== {label} — per-model stats ===")
        print(f"{'Model':<35}{'#Users':>8}{'#Convs':>8}{'#Msgs':>8}{'#Thoughts':>11}")
        print("-" * 70)
        for model in models:
            nu = len(users_by_model[model])
            nc = convs_by_model[model]
            nm = msgs_by_model[model]
            nt = thoughts_by_model[model]
            print(f"{model:<35}{nu:>8,}{nc:>8,}{nm:>8,}{nt:>11,}")
        print("-" * 70)
        print(f"{'Total':<35}{len(total_users):>8,}{total_convs:>8,}{total_msgs:>8,}{total_thoughts:>11,}")
        print()
    else:
        print(f"=== {label} ===")
        print(f"  users:         {len(total_users):>7,}")
        print(f"  conversations: {total_convs:>7,}")
        print(f"  messages:      {total_msgs:>7,}")
        print(f"  thoughts:      {total_thoughts:>7,}")
        print(f"  llms:          {len(convs_by_model):>7,}")

In [3]:
# Option 1: load from HuggingFace
from datasets import load_dataset
ds = load_dataset("SCAI-JHU/ThoughtTrace", split="train")
data = {row["id"]: row for row in ds}

# Option 2: load from JSONL
# data = {}
# with open("data/ThoughtTrace.jsonl") as f:
#     for line in f:
#         row = json.loads(line)
#         data[row["id"]] = row

print_dataset_stats(data, "ThoughtTrace")
print()
print_dataset_stats(data, "ThoughtTrace", mode="modelwise")

=== ThoughtTrace ===
  users:           1,058
  conversations:   2,155
  messages:       17,058
  thoughts:       10,174
  llms:               20

=== ThoughtTrace — per-model stats ===
Model                                #Users  #Convs   #Msgs  #Thoughts
----------------------------------------------------------------------
GPT-5.4                                 162     337   2,462      1,474
Gemini 3.1 Pro Preview                  155     313   2,568      1,553
Grok 4.20                               100     210   1,782        905
Claude Opus 4.6                          70     141   1,222        712
Claude Sonnet 4.6                        68     134   1,224        709
MiniMax M2.7                             50     100     608        344
gpt-oss-120b                             36      70     372        232
Kimi K2.5                                35      71     552        382
Gemma 4 26B A4B                          35      69     504        342
Qwen3.6 Plus                     